Load the Data:

In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.join(os.getcwd(), "../"))

In [3]:
import pandas as pd
import numpy as np

In [4]:
from datetime import datetime

In [5]:
from src.python.utils import load_file_chunks

In [6]:
DATA_PATH = os.path.join(os.getcwd(), "../", "data")
OPTIONS_DATA_PATH = os.path.join(DATA_PATH, "spx_options.csv")
UNDERLYING_DATA_PATH = os.path.join(DATA_PATH, "spx_underlying.csv")

In [7]:
start_date = datetime(2010, 1, 1)
end_date = datetime(2010, 1, 7)

In [ ]:
options_df = load_file_chunks(OPTIONS_DATA_PATH, start_date, end_date)

In [ ]:
underlying_df = load_file_chunks(UNDERLYING_DATA_PATH, start_date, end_date)

Inspect the Data:

In [ ]:
print(options_df.shape)

In [ ]:
print(options_df.dtypes)

In [ ]:
print(options_df.isnull().sum())

In [ ]:
print(options_df.head())

In [ ]:
print(underlying_df.shape)

In [ ]:
print(underlying_df.dtypes)

In [ ]:
print(underlying_df.isnull().sum())

In [ ]:
print(underlying_df.head())

Drop Redundant Columns:

In [ ]:
columns = ['am_set_flag', 'am_settlement', 'ss_flag', 'index_flag', 'issuer', 'secid']

In [ ]:
options_df = options_df.drop(columns=columns)

Drop Rows with Null Greeks:

In [ ]:
greeks_columns = ['impl_volatility', 'delta', 'gamma', 'vega', 'theta']

In [ ]:
all_null = options_df[greeks_columns].isna().all().all()
print(all_null)

In [ ]:
options_df = options_df.dropna(subset=greeks_columns, how='any')

Standardise Data:

In [ ]:
# Standardise Strike Price
options_df['strike_price'] = options_df['strike_price'] / 1000

In [ ]:
# Standardise Expiry Date
options_df['exdate'] = pd.to_datetime(options_df['exdate'], format='%Y-%m-%d')

In [ ]:
# Days to Expiration
options_df['dte'] = (options_df['exdate'] - options_df['date']).dt.days

In [ ]:
options_df['mid'] = ((options_df['best_bid'] + options_df['best_offer']) / 2)

In [ ]:
options_df['bid_ask_spread'] = (options_df['best_offer'] - options_df['best_bid'])

Check Options Exercise Style:

In [ ]:
options_df['exercise_style'].unique()

In [ ]:
# Drop Exercise Style - All European, no Variance
options_df = options_df.drop(columns=['exercise_style'])

In [ ]:
options_df.head()

In [ ]:
print(options_df.shape)

Clean Underlying Data:

In [ ]:
underlying_df = underlying_df.drop(columns=['secid'])

In [ ]:
underlying_df = underlying_df.rename(columns={'close': 'spot', 'return': 'spx_return'})

Merge Dataframes:

In [ ]:
options_df = options_df.merge(underlying_df[['date', 'spot', 'spx_return']], on='date', how='left')

Calculate Moneyness:

In [ ]:
options_df['moneyness'] = options_df['strike_price'] / options_df['spot']

In [ ]:
options_df['log_moneyness'] = np.log(options_df['strike_price'] / options_df['spot'])

In [ ]:
options_df.head()

Save New Dataframe:

In [ ]:
options_df.to_csv(os.path.join(DATA_PATH, "spx_options_processed.csv", index=False))